# 05 — Inference and prediction saving

**Purpose:** Load a saved LoRA adapter or a fully tuned model, generate one category per example, and save raw outputs and metadata.

**Platform:** Kaggle or Colab (Python + GPU for model stages). Never use held-out test data for model selection.

**Start here:** The first code cell clones the public GitHub repository automatically when needed. Enable Internet in Kaggle/Colab.

In [ ]:
# GitHub setup: run this as the FIRST code cell in Kaggle or Colab.
# This clones the source code only; GPU checkpoints/results need separate storage.
from pathlib import Path
import os, sys, subprocess

REPO_URL = "https://github.com/muhammad-omer-dev/nlp_research_proj.git"
REPO_FOLDER = "nlp_research_proj"
PROJECT_FOLDER = "Urdu_Qwen_Research_Notebooks"
PULL_LATEST = False  # True: update an existing clean clone from GitHub

if Path("/kaggle/working").is_dir():
    workspace = Path("/kaggle/working")
elif Path("/content").is_dir():
    workspace = Path("/content")
else:
    workspace = Path.cwd()

repo_dir = workspace / REPO_FOLDER
project_dir = repo_dir / PROJECT_FOLDER

# Local development support: use the existing source folder outside Kaggle/Colab.
local_project = next((p for p in [Path.cwd(), *Path.cwd().parents]
                      if (p / "project_utils.py").is_file()), None)
if local_project is not None and not (Path("/kaggle/working").is_dir() or Path("/content").is_dir()):
    ROOT = local_project
else:
    if not repo_dir.exists():
        print("Cloning:", REPO_URL)
        subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, str(repo_dir)])
    elif PULL_LATEST:
        # Only fast-forward; protects existing local changes from being overwritten.
        subprocess.check_call(["git", "-C", str(repo_dir), "pull", "--ff-only"])
    else:
        print("Using existing clone:", repo_dir)

    ROOT = project_dir

if not (ROOT / "project_utils.py").is_file():
    raise FileNotFoundError(
        f"Cannot find project_utils.py at {ROOT}. Confirm the GitHub folder name."
    )

os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from project_utils import *
folders()
print("Project loaded from:", ROOT)


Run once per trained checkpoint and language. **Only change `SPLIT` to `test` after finalizing training settings.** Run the English test on the same checkpoint if examining retention. The baseline model may use this notebook with `METHOD="base"`.

In [ ]:
import torch
TASK='sib200'; LANG='ur'; SPLIT='validation'  # finally: 'test'
METHOD='lora'  # 'lora', 'full', or 'base'
RUN_ID='sib200_lora_ur_n10_s42_r8'
MAX_ROWS=None  # 10 for smoke, None for full split
labels = task_labels(TASK)
rows = load_task(TASK,LANG,SPLIT)
if MAX_ROWS is not None: rows=rows[:MAX_ROWS]
model_id=cfg()['model_id']; checkpoint=None
if METHOD!='base':
    state=read_json(ROOT/'outputs/checkpoints'/RUN_ID/'complete.json')
    if state['method'] != METHOD or state['task'] != TASK: raise ValueError('Checkpoint metadata does not match method/task')
    model_id=state['base_model']; checkpoint=Path(state['saved_path'])
    if not checkpoint.exists(): checkpoint=ROOT/'outputs/checkpoints'/RUN_ID/'model'
    if not checkpoint.exists(): raise FileNotFoundError(checkpoint)
    if METHOD=='full':
        model,tokenizer=load_model(str(checkpoint))
    else:
        model,tokenizer=load_model(model_id,adapter_path=str(checkpoint))
else:
    model,tokenizer=load_model(model_id)
print('Model:',model_id,'mode:',METHOD,'rows:',len(rows))

In [ ]:
path=ROOT/'outputs/predictions'/f'{RUN_ID}_{LANG}_{SPLIT}.csv'
meta=dict(task=TASK,lang=LANG,split=SPLIT,model_id=model_id,
          run_id=RUN_ID,method=METHOD,checkpoint=str(checkpoint) if checkpoint else None,
          dataset_sha256=dataset_sha(TASK,LANG,SPLIT),subset=(MAX_ROWS is not None),
          decoding={'do_sample':False,'max_new_tokens':cfg()['max_new_tokens']})
outputs=infer_resume(model,tokenizer,rows,labels,LANG,path,meta,cfg()['max_new_tokens'])
print('Saved:',path, 'count:',len(outputs),'invalid:',sum(not x['valid'] for x in outputs))
print('First outputs:',[(p['reference'],p['prediction'],p['raw_prediction']) for p in outputs[:8]])

In [ ]:
del model
if torch.cuda.is_available(): torch.cuda.empty_cache()